# Exact Schwarzian transport for (X(14,1)), (X(15,1)), and (X(21,1))

Choose the **SageMath** kernel and run the cells in order. This notebook is
self-contained: every calculation uses exact rational arithmetic. Each form has
its own input, model comparison, Schwarzian calculation, and parameter recovery.

We start with Tu's published (Q_N(T)=\tfrac12\{z,T\}), identify her quotient
coordinate (T=\phi(u)) in the González–Rotger model, and compute
$\mathcal R_N(u)=\{z,t\}$, where $dt=dx/y$. We then recover (L,u_1=u'(0))
from the supplied CM value (u_0) and the exact initial identities (c_2,c_3).
These CM markings and coefficient identities are **inputs**, not proved by this
notebook. Once they are supplied, the parameter recovery and subsequent
coefficients are exact; no numerical recognition is used.

To keep the two coordinates distinct, **(T)** always denotes Tu's quotient
coordinate and **(t)** the local integral of (dx/y). We write the source
quartic coordinates as ((s,v)), renaming the vertical coordinate in Elkies and Tu.
Our quartic coordinates are ((x,y)), with (u=x^2).


## The conversion formula

We use the classical Schwarzian
$$
\{f,u\}=\frac{f'''(u)}{f'(u)}-\frac32\left(\frac{f''(u)}{f'(u)}\right)^2.
$$
Tu defines (Q(T)=-\{T,z\}/(2T_z^2)=\tfrac12\{z,T\})
([Tu, §2, p. 455](https://msp.org/pjm/2014/269-2/pjm-v269-n2-p14-s.pdf)).
Thus the factor (2) in the following calculation is essential.

For (y^2=G(u)), (u=x^2), and (dt=dx/y), we have
$$
u_t=2xy,\qquad u_t^2=H(u):=4uG(u),\qquad
u_{tt}=\frac12H'(u),\qquad u_{ttt}=\frac12H''(u)u_t.
$$
Applying the chain rule first to (z(T(u))), then to (z(u(t))), gives
$$
\begin{aligned}
\mathcal R(u)=\{z,t\}
 &=\bigl(2Q(\phi(u))\phi'(u)^2+\{\phi,u\}\bigr)H(u)+\{u,t\}\\
 &=2H(u)Q(\phi(u))\phi'(u)^2+H(u)\{\phi,u\}
   +\frac12H''(u)-\frac{3H'(u)^2}{8H(u)}.
\end{aligned}
$$
Here primes on (H,\phi,\mathcal R) mean (d/du). The calculation is initially
local where (u_t\ne0), and the resulting rational identity extends
meromorphically. All three functions $\phi$ below are affine, so
$\{\phi,u\}=0$; the code nevertheless computes this term.

The source model changes preserve (dx/y=ds/v). This check prevents an
unintended change in the normalization of (t), and hence of $\mathcal R$.


In [1]:
from sage.all import QQ, PolynomialRing, PowerSeriesRing, factorial, latex
from IPython.display import display, Math, Markdown

Px = PolynomialRing(QQ, 'x'); x = Px.gen()
Ps = PolynomialRing(QQ, 's'); s = Ps.gen()
PT = PolynomialRing(QQ, 'T'); T = PT.gen(); FT = PT.fraction_field()
Pu = PolynomialRing(QQ, 'u'); u = Pu.gen(); Fu = Pu.fraction_field()

def show_math(name, value):
    display(Math(name + '=' + latex(value)))

def evaluate(f, argument):
    '''Evaluate a rational function without changing its variable convention.'''
    return f.numerator()(argument) / f.denominator()(argument)

def schwarzian(f):
    f = Fu(f)
    assert f.derivative() != 0
    return f.derivative(3)/f.derivative() - (QQ(3)/2)*(f.derivative(2)/f.derivative())**2

def replace_square(polynomial, square):
    '''For an even polynomial in s or x, replace s^2 or x^2 by square.'''
    assert all(polynomial[j] == 0 for j in range(1, polynomial.degree()+1, 2))
    return sum(polynomial[2*j]*square**j for j in range(polynomial.degree()//2+1))

def compare_models(case):
    alpha, beta = case['x_scale'], case['y_scale']
    difference = Ps(case['ours'](alpha*s) - beta**2*case['source'])
    assert difference == 0
    assert alpha/beta == 1
    case['H'] = Pu(4*u*replace_square(case['ours'], u))
    case['phi'] = Fu(replace_square(case['source_T'], u/alpha**2))
    assert evaluate(case['phi'], (alpha*s)**2) == case['source_T']
    show_math(r'F_{\rm ours}(\alpha s)-\beta^2F_{\rm source}(s)', difference)
    display(Math(r'\frac{dx}{y}=\frac{\alpha}{\beta}\frac{ds}{v}=\frac{ds}{v}'))
    show_math(r'T=\phi(u)', case['phi'])
    display(Math(r'H_{%s}(u)=4u\bigl(%s\bigr)' %
                 (case['N'], latex(replace_square(case['ours'], u)))))
    print('PASS: the model substitution, differential normalization, and quotient coordinate.')

def transport_schwarzian(case):
    H, phi, Q = case['H'], case['phi'], case['Q']
    schwarz_phi = schwarzian(phi)
    tu_term = Fu(2*H*evaluate(Q, phi)*phi.derivative()**2)
    coordinate_term = Fu(H.derivative(2)/2 - 3*H.derivative()**2/(8*H))
    R = Fu(tu_term + H*schwarz_phi + coordinate_term)
    case['R'] = R
    show_math(r'\phi^{\prime}(u)^2', phi.derivative()**2)
    show_math(r'\{\phi,u\}', schwarz_phi)
    show_math(r'H^{\prime}(u)', H.derivative())
    show_math(r'H^{\prime\prime}(u)', H.derivative(2))
    display(Math(
        r'\mathcal R_{%s}(u)=%s\,H_{%s}(u)Q_{%s}\!\left(%s\right)'
        r'+\frac12H_{%s}^{\prime\prime}(u)-\frac{3H_{%s}^{\prime}(u)^2}{8H_{%s}(u)}'
        % (case['N'], latex(2*phi.derivative()**2), case['N'], case['N'],
           latex(phi), case['N'], case['N'], case['N'])))
    # Display the reduced expression with a factored denominator.
    if R.denominator() == 1:
        show_math(r'\mathcal R_{%s}(u)' % case['N'], R)
    else:
        display(Math(r'\mathcal R_{%s}(u)=\frac{%s}{%s}' %
                     (case['N'], latex(R.numerator()), latex(R.denominator().factor()))))
    assert R - tu_term - H*schwarz_phi - coordinate_term == 0
    print('PASS: exact Schwarzian-chain-rule identity in QQ(u).')
    return R


## Recovering (L,u_1) and computing the next coefficient

Write
$$
A(q)=\sum_{n\ge0}\frac{c_n}{n!}q^n,\qquad c_0=c_1=1,\qquad
u_i=\left.\frac{d^i u}{dq^i}\right|_{q=0}.
$$
The differential normalization is $dt=\lambda A(q)\,dq$, with $L=\lambda^2$.
The equation
$$
AA''-\frac32(A')^2+L\mathcal R(u(q))A^4=0
$$
at (q=0) gives (L\mathcal R(u_0)=3/2-c_2). Its first derivative gives
$$
c_3-2c_2+L\mathcal R'(u_0)u_1+4L\mathcal R(u_0)=0.
$$
Consequently, when the displayed denominators are nonzero,
$$
\boxed{L=\frac{3/2-c_2}{\mathcal R(u_0)},\qquad
u_1=\frac{6c_2-c_3-6}{L\mathcal R'(u_0)}.}
$$
Each example also checks (u_1^2=LH(u_0)). To generate coefficients, we use
$$
A''=\frac{3(A')^2}{2A}-L\mathcal R(u)A^3,\qquad
u''=\frac{A'}{A}u'+\frac L2A^2H'(u).
$$
The coefficients of (q^n) on the right determine those of (q^{n+2}) on the
left by division by ((n+1)(n+2)). Starting only from (A(0)=A'(0)=1) and the
recovered (u_0,u_1,L), the code reproduces (c_2,c_3) and predicts (c_4).


In [2]:
def generate_coefficients(case, last_index=4):
    '''Use the recovered recurrence, without re-inserting the seed coefficients.'''
    Qq = PowerSeriesRing(QQ, 'q', default_prec=last_index+4)
    H, R, L = case['H'], case['R'], case['L']
    a, b = [QQ(1), QQ(1)], [case['u0'], case['u1']]
    for n in range(last_index-1):
        A, U = Qq(a), Qq(b)
        rhs_A = (QQ(3)/2)*A.derivative()**2/A - L*evaluate(R, U)*A**3
        rhs_u = A.derivative()*U.derivative()/A + (L/2)*A**2*H.derivative()(U)
        denominator = (n+1)*(n+2)
        a.append(QQ(rhs_A[n])/denominator)
        b.append(QQ(rhs_u[n])/denominator)
    A, U = Qq(a), Qq(b)
    first_integral = U.derivative()**2 - L*A**2*H(U)
    assert all(first_integral[n] == 0 for n in range(last_index))
    return [QQ(a[n]*factorial(n)) for n in range(last_index+1)]

def recover_parameters(case):
    R, H, u0, c2, c3 = (case[k] for k in ('R', 'H', 'u0', 'c2', 'c3'))
    assert R.denominator()(u0) != 0 and H(u0) != 0
    R0, Rprime0 = QQ(evaluate(R, u0)), QQ(evaluate(R.derivative(), u0))
    assert R0 != 0
    case['L'] = L = (QQ(3)/2-c2)/R0
    assert L*Rprime0 != 0
    case['u1'] = u1 = (6*c2-c3-6)/(L*Rprime0)
    show_math(r'\mathcal R(u_0)', R0)
    show_math(r'\mathcal R^{\prime}(u_0)', Rprime0)
    show_math('L', L)
    show_math('u_1', u1)
    show_math(r'u_1^2-LH(u_0)', u1**2-L*H(u0))
    assert u1**2 == L*H(u0)
    coefficients = generate_coefficients(case)
    assert coefficients[:4] == [1, 1, c2, c3]
    case['coefficients'] = coefficients
    print('PASS: both input coefficient identities and the first integral.')
    display(Markdown('**First coefficient beyond the input range (computed, not assumed):**'))
    show_math('c_4', coefficients[4])


## 14a: Elkies's quartic and (T=-u/16)

[Elkies, §5.1, equation (68), p. 33](https://arxiv.org/pdf/math/0005160) gives
$$
v^2=-16s^4+13s^2-8,\qquad T=-s^2.
$$
The González–Rotger model is
$$
y^2=-x^4+13x^2-128,\qquad u=x^2
$$
([González–Rotger, Theorem 3.4](https://www.jstage.jst.go.jp/article/jmath/58/4/58_4_927/_pdf)).
The change (x=4s,\ y=4v) identifies the models and preserves (dx/y=ds/v).
Hence (T=-s^2=-u/16).

[Tu, §5.1, p. 482](https://msp.org/pjm/2014/269-2/pjm-v269-n2-p14-s.pdf) gives
$$
Q_{14}(T)=\frac{192+440T+43T^2+1036T^3+960T^4}
{16T^2(8+13T+16T^2)^2}.
$$
The CM and coefficient inputs are (D=-11), (u_0=16), (c_2=-4), (c_3=214/9).


In [3]:
case14 = dict(
    N=14, label='14a',
    source=-16*s**4 + 13*s**2 - 8,
    ours=-x**4 + 13*x**2 - 128,
    x_scale=QQ(4), y_scale=QQ(4), source_T=-s**2,
    Q=FT((192 + 440*T + 43*T**2 + 1036*T**3 + 960*T**4)
         / (16*T**2*(8 + 13*T + 16*T**2)**2)),
    D=-11, u0=QQ(16), c2=QQ(-4), c3=QQ(214)/9,
)
compare_models(case14)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: the model substitution, differential normalization, and quotient coordinate.


In [4]:
R14 = transport_schwarzian(case14)
# A compact equivalent form of the computed rational function:
assert R14 == (QQ(3)/8)*(4-u)
display(Math(r'\boxed{\mathcal R_{14}(u)=\frac38(4-u)}'))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: exact Schwarzian-chain-rule identity in QQ(u).


<IPython.core.display.Math object>

In [5]:
recover_parameters(case14)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: both input coefficient identities and the first integral.


**First coefficient beyond the input range (computed, not assumed):**

<IPython.core.display.Math object>

## 15a: the same quartic and (T=-3u)

[Elkies, §5.2, equation (76), p. 35](https://arxiv.org/pdf/math/0005160) gives
$$
v^2=-(3s^2+1)(s^2+27),\qquad T=-3s^2.
$$
Expanding the product gives the González–Rotger model
$$
y^2=-3x^4-82x^2-27,\qquad u=x^2
$$
([González–Rotger, Theorem 3.4](https://www.jstage.jst.go.jp/article/jmath/58/4/58_4_927/_pdf)).
Thus (x=s,\ y=v), (dx/y=ds/v), and (T=-3u).

[Tu, §5.1, p. 482](https://msp.org/pjm/2014/269-2/pjm-v269-n2-p14-s.pdf) gives
$$
Q_{15}(T)=\frac{177147-244944T+244242T^2-3680T^3+35T^4}
{144T^2(1-T)^2(81-T)^2}.
$$
The CM and coefficient inputs are (D=-7), (u_0=9), (c_2=-1/4), (c_3=27/16).


In [6]:
case15 = dict(
    N=15, label='15a',
    source=-(3*s**2 + 1)*(s**2 + 27),
    ours=-3*x**4 - 82*x**2 - 27,
    x_scale=QQ(1), y_scale=QQ(1), source_T=-3*s**2,
    Q=FT((177147 - 244944*T + 244242*T**2 - 3680*T**3 + 35*T**4)
         / (144*T**2*(1-T)**2*(81-T)**2)),
    D=-7, u0=QQ(9), c2=-QQ(1)/4, c3=QQ(27)/16,
)
compare_models(case15)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: the model substitution, differential normalization, and quotient coordinate.


In [7]:
R15 = transport_schwarzian(case15)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: exact Schwarzian-chain-rule identity in QQ(u).


In [8]:
recover_parameters(case15)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: both input coefficient identities and the first integral.


**First coefficient beyond the input range (computed, not assumed):**

<IPython.core.display.Math object>

## 21a: Tu's quotient coordinate is (T=u)

The González–Rotger quartic, also used in
[Tu, proof of Lemma 16, p. 470](https://msp.org/pjm/2014/269-2/pjm-v269-n2-p14-s.pdf), is
$$
v^2=-7s^4+94s^2-343.
$$
Our model is (y^2=-7x^4+94x^2-343), so (x=s,\ y=v).
In that proof Tu passes to the cubic quotient by
$$
(s,v)\longmapsto(T,Y)=(s^2,sv),\qquad
Y^2=T(-7T^2+94T-343),
$$
and takes its horizontal coordinate as the quotient parameter.
Therefore **(T=s^2=x^2=u)**. The code checks the cubic equation as well as the
identity of the quartics. The differential used here remains (dx/y=ds/v)
on the original quartic.

[Tu, §5.1, p. 483](https://msp.org/pjm/2014/269-2/pjm-v269-n2-p14-s.pdf) gives
$$
Q_{21}(T)=\frac{21\bigl(40353607-17647350T+3561369T^2-477652T^3
                      +31833T^4-630T^5+7T^6\bigr)}
{16T^2(49-T)^2(343-94T+7T^2)^2}.
$$
We supply the CM marking (D=-43), (u_0=441/25), and the exact initial
identities (c_2=-905/3969), (c_3=-1318/27783). This CM marking is a separate
geometric input; it is not a consequence of the quoted formula for (Q_{21}).


In [9]:
case21 = dict(
    N=21, label='21a',
    source=-7*s**4 + 94*s**2 - 343,
    ours=-7*x**4 + 94*x**2 - 343,
    x_scale=QQ(1), y_scale=QQ(1), source_T=s**2,
    Q=FT(21*(40353607 - 17647350*T + 3561369*T**2 - 477652*T**3
             + 31833*T**4 - 630*T**5 + 7*T**6)
         / (16*T**2*(49-T)**2*(343-94*T+7*T**2)**2)),
    D=-43, u0=QQ(441)/25, c2=-QQ(905)/3969, c3=-QQ(1318)/27783,
)
compare_models(case21)
quotient_cubic = T*(-7*T**2 + 94*T - 343)
assert s**2*case21['source'] == quotient_cubic(s**2)
show_math('Y^2', quotient_cubic)
print('PASS: the cubic quotient has horizontal coordinate T=s^2=u.')


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: the model substitution, differential normalization, and quotient coordinate.


<IPython.core.display.Math object>

PASS: the cubic quotient has horizontal coordinate T=s^2=u.


In [10]:
R21 = transport_schwarzian(case21)
# Partial fractions expose the constant and the two terms over the elliptic fiber.
polynomial_part, proper_parts = R21.partial_fraction_decomposition()
assert Fu(polynomial_part + sum(proper_parts)) == R21
display(Math(r'\mathcal R_{21}(u)=' + latex(polynomial_part)
             + ''.join(r'+\left(%s\right)' % latex(part) for part in proper_parts)))
assert R21 == -272 - 32928/(u-49) - 921984/(u-49)**2
display(Math(r'\boxed{\mathcal R_{21}(u)=-272-\frac{32928}{u-49}'
             r'-\frac{921984}{(u-49)^2}}'))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: exact Schwarzian-chain-rule identity in QQ(u).


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [11]:
recover_parameters(case21)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS: both input coefficient identities and the first integral.


**First coefficient beyond the input range (computed, not assumed):**

<IPython.core.display.Math object>

## Computed summary and independent answer checks

The summary below is assembled from the preceding computations. The final
assertions compare them with the values in the draft; those comparison values
were not used to recover (L,u_1) or to generate any coefficient.


In [12]:
cases = [case14, case15, case21]
rows = [r'\begin{array}{c|c|c|c|c|c}'
        r'\text{Form}&\phi(u)&u_0&L&u_1&c_4\ \text{(derived)}\\\hline']
for case in cases:
    rows.append(r'\text{%s}&%s&%s&%s&%s&%s\\' %
                (case['label'], latex(case['phi']), latex(case['u0']),
                 latex(case['L']), latex(case['u1']), latex(case['coefficients'][4])))
rows.append(r'\end{array}')
display(Math('\n'.join(rows)))

reference_values = {
    '14a': ((QQ(3)/8)*(4-u), -QQ(11)/9, -QQ(352)/3, QQ(24)),
    '15a': (-4-(QQ(4)/3)*u, -QQ(7)/64, QQ(-63), -QQ(381)/64),
    '21a': (-16*(17*u**2+392*u-2401)/(u-49)**2,
            -QQ(43)/3969, -QQ(9632)/375, -QQ(4357004)/5250987),
}
for case in cases:
    R, L, u1, c4 = reference_values[case['label']]
    assert case['R'] == R
    assert (case['L'], case['u1'], case['coefficients'][4]) == (L, u1, c4)
print('PASS: all three Schwarzian functions, six recovered parameters, and three predictions agree exactly.')

# Optional: compute more coefficients using the same exact recurrence.
# generate_coefficients(case14, last_index=10)


<IPython.core.display.Math object>

PASS: all three Schwarzian functions, six recovered parameters, and three predictions agree exactly.


## References and conventions

1. **Fang-Ting Tu**, *Schwarzian differential equations associated to Shimura
   curves of genus zero*, Pacific J. Math. **269** (2014), 453–489.
   [Article PDF](https://msp.org/pjm/2014/269-2/pjm-v269-n2-p14-s.pdf),
   [DOI](https://doi.org/10.2140/pjm.2014.269.453).
   The convention for (Q) is in §2, p. 455; the three displayed rational
   functions are in §5.1, pp. 482–483. Lemmas 12–13 describe the quotient
   markings for 14 and 15; the proof of Lemma 16, p. 470, identifies the
   quotient coordinate for 21.
2. **Noam D. Elkies**, *Shimura curve computations*, in *Algorithmic Number
   Theory (Portland, OR, 1998)*, Lecture Notes in Computer Science **1423**,
   Springer, 1998, 1–47.
   [arXiv version](https://arxiv.org/abs/math/0005160),
   [PDF](https://arxiv.org/pdf/math/0005160).
   We use the models and quotient coordinates in §5.1, equation (68), and
   §5.2, equation (76), on pp. 33 and 35 of the arXiv PDF. Elkies uses a
   Schwarzian convention twice the classical one; this notebook uses
   **Tu's (Q=\tfrac12\{z,T\})** throughout.
3. **Josep González and Victor Rotger**, *Non-elliptic Shimura curves of genus
   one*, J. Math. Soc. Japan **58** (2006), 927–948, Theorem 3.4.
   [Article PDF](https://www.jstage.jst.go.jp/article/jmath/58/4/58_4_927/_pdf),
   [DOI](https://doi.org/10.2969/jmsj/1179759530).
   This is the source for the three models used in the draft.

The published projective connections determine $\mathcal R$ independently
of the assumed initial coefficient identities. The latter, together with the
specified (u_0), determine (L,u_1); subsequent (c_n) then follow from the
coupled recurrence by rational arithmetic.
